In [ ]:
# Install required libraries in the current notebook kernel
%pip install mlflow==2.12.2 scikit-learn

In [1]:
# Imports for model training + MLflow logging
import os
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

/opt/conda/lib/python3.11/site-packages/mlflow/utils/requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251


In [2]:
# Point MLflow client to your OKE-hosted tracking server
tracking_uri = os.getenv("MLFLOW_TRACKING_URI", "http://mlflow.mlflow.svc.cluster.local")
mlflow.set_tracking_uri(tracking_uri)
# Experiment groups related runs together in the UI
mlflow.set_experiment("basic-iris-training")
# Remove bad env override if present
os.environ.pop("MLFLOW_TRACKING_URI", None)

# Use your OKE MLflow server explicitly
mlflow.set_tracking_uri("http://mlflow.mlflow.svc.cluster.local")
print("Tracking URI:", mlflow.get_tracking_uri())


2026/03/17 19:57:02 INFO mlflow.tracking.fluent: Experiment with name 'basic-iris-training' does not exist. Creating a new experiment.


Tracking URI: http://mlflow.mlflow.svc.cluster.local


In [3]:
# Load a tiny built-in dataset and split into train/test
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# Train a simple baseline classifier
model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)
preds = model.predict(X_test)
# Basic evaluation metric
acc = accuracy_score(y_test, preds)
print("Test Accuracy:", round(acc, 4))

Test Accuracy: 0.9667


In [8]:
# Log training details, metric, and model artifact in one MLflow run
with mlflow.start_run(run_name="iris-logreg-baseline"):
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("max_iter", 200)
    mlflow.log_metric("test_accuracy", float(acc))
    mlflow.sklearn.log_model(model, artifact_path="model")
    run = mlflow.active_run()
    print("Run ID:", run.info.run_id)
    print("Logged accuracy:", round(acc, 4))

Run ID: 4eacf1a845d9452790160139fd89f58d
Logged accuracy: 0.9667
